# 13 — Evaluación de los modelos exportados (Keras, float32 e int8)

**Objetivo.** Cuantificar la pérdida de exactitud introducida por la exportación a TensorFlow Lite y por la
cuantización a enteros de 8 bits, comparando las tres variantes de cada clasificador sobre el mismo conjunto de
prueba independiente.

**Tecnología.** TensorFlow/Keras y el intérprete de TensorFlow Lite para la inferencia; scikit-learn para las
métricas. No se realiza entrenamiento: se evalúan los modelos ya exportados.

**Metodología.** Para cada imagen del conjunto de prueba se construye una sola vez la doble entrada del sistema
(imagen normalizada con Shades-of-Gray e imagen de hoja aislada mediante la máscara de M_seg) y se presenta sin
modificación a las tres variantes. Los modelos EfficientNet reciben valores en el rango [0, 255], ya que la
normalización está incluida en la propia red. En la variante entera, las entradas y salidas se convierten mediante
los parámetros de cuantización declarados por el intérprete. Se reportan exactitud y F1 macro por variante y su
diferencia respecto del modelo Keras de referencia.

In [ ]:
!pip install -q pycocotools opencv-python-headless

In [ ]:
from pathlib import Path
import glob
from google.colab import drive

drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/glycine_vision_baselines')
OUT.mkdir(parents=True, exist_ok=True)
DRIVE = Path('/content/drive/MyDrive')

IMPRESCINDIBLES = [
    ('m1_keras', 'model1_binary.keras', 'notebook 03'),
    ('m1_f32', 'model1.tflite', 'notebook 06'),
    ('m1_int8', 'model1_int8.tflite', 'notebook 06'),
    ('m2_keras', 'model2_pathogen.keras', 'notebook 04'),
    ('m2_f32', 'model2.tflite', 'notebook 06'),
    ('m2_int8', 'model2_int8.tflite', 'notebook 06'),
    ('seg_keras', 'model_seg.keras', 'notebook 02'),
]
SOLO_SEGMENTADOR = [
    ('seg_f32', 'model_seg.tflite', 'notebook 06'),
    ('seg_int8', 'model_seg_int8.tflite', 'notebook 06'),
]


def _first(candidatos, patron=None):
    for c in candidatos:
        if Path(c).exists():
            return Path(c)
    if patron:
        hits = sorted(glob.glob(patron, recursive=True))
        if hits:
            return Path(hits[0])
    return None


SPLIT = _first(['/content/splits'], str(DRIVE / '**' / 'splits'))
assert SPLIT is not None, 'No se encontro la carpeta splits.'
TEST_BIN = _first([SPLIT / 'test' / 'clasificacion_binaria'])
TEST_PAT = _first([SPLIT / 'test' / 'clasificacion_patogeno'])
MASKS = _first([SPLIT / 'masks'], str(SPLIT / '**' / 'masks'))
MASKS_SOY = _first([SPLIT / 'masks_soycotton'], str(SPLIT / '**' / 'masks_soycotton'))

MODELOS = {}
for clave, nombre, _ in IMPRESCINDIBLES + SOLO_SEGMENTADOR:
    MODELOS[clave] = _first([OUT / nombre], str(DRIVE / '**' / nombre))
SEG_PATH = MODELOS['seg_keras']

print('=' * 76)
print('INVENTARIO DE ENTRADAS')
print('=' * 76)
print(f'particiones   : {SPLIT}')
for etiqueta, ruta in [('test binario', TEST_BIN), ('test patogeno', TEST_PAT),
                       ('masks', MASKS), ('masks_soycotton', MASKS_SOY)]:
    print(f'{etiqueta:14s}: {ruta if ruta else "NO ENCONTRADO"}')
for clave, nombre, origen in IMPRESCINDIBLES + SOLO_SEGMENTADOR:
    ruta = MODELOS[clave]
    print(f'{nombre:22s}: {ruta if ruta else "NO ENCONTRADO -> lo genera el " + origen}')

bloqueantes = [n for c, n, _ in IMPRESCINDIBLES if MODELOS[c] is None]
bloqueantes += [e for e, r in [('test/clasificacion_binaria', TEST_BIN),
                               ('test/clasificacion_patogeno', TEST_PAT)] if r is None]
if bloqueantes:
    print('\nFalta lo indispensable. model_seg.keras tambien hace falta aqui: genera la hoja aislada')
    print('que M1 y M2 reciben como segunda entrada.')
    for nombre in bloqueantes:
        print(f'   {nombre}')
    raise AssertionError('Entradas incompletas: ' + ', '.join(bloqueantes))

pendientes_seg = [n for c, n, _ in SOLO_SEGMENTADOR if MODELOS[c] is None]
pendientes_seg += [e for e, r in [('masks/', MASKS), ('masks_soycotton/', MASKS_SOY)] if r is None]
SEG_DISPONIBLE = not pendientes_seg
if SEG_DISPONIBLE:
    print('\nTodo presente: se evaluaran los clasificadores y el segmentador.')
else:
    print('\nAVISO: se omitiran las celdas del segmentador porque faltan:')
    for nombre in pendientes_seg:
        print(f'   {nombre}')
    print('M1 y M2 si se evaluaran. La observacion 2 del informe necesita el segmentador.')

In [ ]:
import json
import numpy as np
import cv2
import tensorflow as tf
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score

_MSEG = tf.keras.models.load_model(SEG_PATH, compile=False)


def chromatic_normalize(img_rgb):
    x = img_rgb.astype(np.float32)
    il = np.power(np.mean(np.power(x, 6), axis=(0, 1)), 1.0 / 6.0)
    return np.clip(x * np.clip(il.mean() / (il + 1e-6), 0.6, 1.6), 0, 255).astype(np.uint8)


def mseg_mask(img_rgb, size):
    small = chromatic_normalize(cv2.resize(img_rgb, (256, 256)))
    prob = _MSEG.predict((small.astype(np.float32) / 255.0)[np.newaxis], verbose=0)[0]
    leaf = (np.argmax(prob, -1) == 1).astype(np.uint8)
    return cv2.resize(leaf, size, interpolation=cv2.INTER_NEAREST)


def construir_entradas(directorio, size):
    clases = sorted(p.name for p in Path(directorio).iterdir() if p.is_dir())
    idx = {c: i for i, c in enumerate(clases)}
    originales, hojas, etiquetas = [], [], []
    for c in clases:
        for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp'):
            for fp in (Path(directorio) / c).glob(ext):
                img = np.array(Image.open(fp).convert('RGB').resize(size))
                norm = chromatic_normalize(img)
                iso = norm.copy()
                iso[mseg_mask(img, size) == 0] = 0
                originales.append(norm)
                hojas.append(iso)
                etiquetas.append(idx[c])
    return clases, np.stack(originales), np.stack(hojas), np.array(etiquetas)

In [ ]:
HILOS = 4


def predecir_keras(ruta, Xo, Xl, batch=16):
    modelo = tf.keras.models.load_model(ruta, compile=False)
    salidas = modelo.predict([Xo.astype(np.float32), Xl.astype(np.float32)], batch_size=batch, verbose=0)
    tf.keras.backend.clear_session()
    return salidas


def _cuantizar(x, detalle):
    escala, cero = detalle['quantization']
    if escala == 0:
        return x.astype(detalle['dtype'])
    limites = np.iinfo(detalle['dtype'])
    return np.clip(np.round(x / escala + cero), limites.min, limites.max).astype(detalle['dtype'])


def _descuantizar(y, detalle):
    escala, cero = detalle['quantization']
    if escala == 0:
        return y.astype(np.float32)
    return (y.astype(np.float32) - cero) * escala


def predecir_tflite(ruta, Xo, Xl):
    interp = tf.lite.Interpreter(model_path=str(ruta), num_threads=HILOS)
    interp.allocate_tensors()
    entradas = interp.get_input_details()
    salida = interp.get_output_details()[0]
    claves = ('hoja', 'aislada', 'leaf')
    resultados = []
    for i in range(len(Xo)):
        for detalle in entradas:
            fuente = Xl[i] if any(k in detalle['name'].lower() for k in claves) else Xo[i]
            dato = fuente.astype(np.float32)[np.newaxis]
            if detalle['dtype'] in (np.uint8, np.int8):
                dato = _cuantizar(dato, detalle)
            interp.set_tensor(detalle['index'], dato.astype(detalle['dtype']))
        interp.invoke()
        resultados.append(_descuantizar(interp.get_tensor(salida['index'])[0], salida))
    return np.stack(resultados)


def metricas(probs, y, binario):
    pred = (probs.reshape(-1) >= 0.5).astype(int) if binario else probs.argmax(axis=1)
    return accuracy_score(y, pred), f1_score(y, pred, average='macro', zero_division=0)

In [ ]:
import pandas as pd

filas = []
for etiqueta, test_dir, size, binario, claves in [
        ('M1 (estado sanitario)', TEST_BIN, (240, 240), True, ('m1_keras', 'm1_f32', 'm1_int8')),
        ('M2 (patogeno)', TEST_PAT, (224, 224), False, ('m2_keras', 'm2_f32', 'm2_int8'))]:
    clases, Xo, Xl, y = construir_entradas(test_dir, size)
    print(f'{etiqueta}: {len(y)} imagenes | clases {clases}')
    if binario:
        pos = next((i for i, c in enumerate(clases) if 'enferm' in c.lower()), 1)
        y_eval = (y == pos).astype(int)
    else:
        pos = None
        y_eval = y
    referencia = None
    for variante, clave in zip(('Keras', 'TFLite float32', 'TFLite int8'), claves):
        ruta = MODELOS[clave]
        probs = predecir_keras(ruta, Xo, Xl) if clave.endswith('keras') else predecir_tflite(ruta, Xo, Xl)
        if binario and probs.shape[-1] == 1 and pos == 0:
            probs = 1.0 - probs
        acc, f1 = metricas(probs, y_eval, binario)
        if referencia is None:
            referencia = (acc, f1)
        filas.append({'Modelo': etiqueta, 'Variante': variante,
                      'Tamano (MB)': round(Path(ruta).stat().st_size / 1e6, 2),
                      'Exactitud': round(acc, 4), 'F1 macro': round(f1, 4),
                      'D Exactitud': round(acc - referencia[0], 4),
                      'D F1 macro': round(f1 - referencia[1], 4)})
        print(f'  {variante:15s} exactitud={acc:.4f}  F1={f1:.4f}')

tabla = pd.DataFrame(filas)
tabla.to_csv(OUT / 'evaluacion_variantes.csv', index=False)
json.dump(filas, open(OUT / 'evaluacion_variantes.json', 'w'), indent=2, ensure_ascii=False)
tabla

## Segmentador: efecto de la exportacion y la cuantizacion

M_seg se despliega en **int8** tanto en la aplicacion como en el servicio de respaldo, pero hasta aqui solo se
habian reportado sus metricas en Keras. Esta seccion evalua las tres variantes sobre el **mismo 25 % reservado**
de las mascaras COCO fusionadas que emplea el notebook `05` (fraccion 0.25, semilla 42), de modo que las cifras
son directamente comparables con las de la Tabla 1.

Se reportan las tres metricas del segmentador: **recall de hoja** (la principal, porque un falso negativo elimina
tejido que las etapas siguientes ya no pueden recuperar), **Dice** e **IoU**. La conversion de entradas y salidas
usa los parametros de cuantizacion declarados por el interprete, igual que para M1 y M2.


In [ ]:
if not SEG_DISPONIBLE:
    print('Seccion del segmentador omitida: faltan entradas (ver el inventario de la celda 3).')
else:
    from pycocotools.coco import COCO

    MASK_SOURCES = []
    for base, subdir in [(MASKS, MASKS.name if MASKS else 'masks'),
                         (MASKS_SOY, f'{MASKS_SOY.name}/images' if MASKS_SOY else 'masks_soycotton/images')]:
        if base is None:
            continue
        candidatos = sorted(base.rglob('*.json'))
        if candidatos:
            MASK_SOURCES.append((candidatos[0], subdir))
    assert MASK_SOURCES, 'No se encontraron anotaciones COCO dentro de masks/ ni masks_soycotton/'
    RAIZ_MASCARAS = MASKS.parent if MASKS else MASKS_SOY.parent

    def cargar_coco_fusionado(fuentes):
        imagenes, anotaciones = [], []
        iid, aid, mapa = 0, 0, {}
        for si, (ruta, subdir) in enumerate(fuentes):
            datos = json.load(open(ruta, encoding='utf-8'))
            for im in datos['images']:
                ni = dict(im)
                ni['id'] = iid
                ni['file_name'] = f"{subdir}/{Path(im['file_name']).name}"
                mapa[(si, im['id'])] = iid
                imagenes.append(ni)
                iid += 1
            for an in datos['annotations']:
                clave = (si, an['image_id'])
                if clave not in mapa or not an.get('segmentation'):
                    continue
                na = dict(an)
                na['id'] = aid
                na['image_id'] = mapa[clave]
                na['category_id'] = 1
                anotaciones.append(na)
                aid += 1
        coco = COCO()
        coco.dataset = {'images': imagenes, 'annotations': anotaciones,
                        'categories': [{'id': 1, 'name': 'hoja', 'supercategory': 'leaf'}]}
        coco.createIndex()
        return coco

    def ids_reservados(coco, fraccion=0.25, semilla=42):
        ids = sorted(coco.getImgIds())
        np.random.RandomState(semilla).shuffle(ids)
        return ids[:round(len(ids) * fraccion)]

    def verdad_de_campo(coco, img_id, size=(256, 256)):
        info = coco.loadImgs(img_id)[0]
        hoja = np.zeros((info['height'], info['width']), np.uint8)
        for ann in coco.loadAnns(coco.getAnnIds(imgIds=img_id)):
            hoja = np.maximum(hoja, coco.annToMask(ann))
        return info, cv2.resize(hoja, size, interpolation=cv2.INTER_NEAREST)

    coco = cargar_coco_fusionado(MASK_SOURCES)
    IDS_TEST = ids_reservados(coco)

    ENTRADAS_SEG, VERDAD_SEG, sin_archivo = [], [], 0
    for img_id in IDS_TEST:
        info, gt = verdad_de_campo(coco, img_id)
        fp = RAIZ_MASCARAS / info['file_name']
        if not fp.exists():
            sin_archivo += 1
            continue
        crudo = np.array(Image.open(fp).convert('RGB').resize((256, 256)))
        ENTRADAS_SEG.append(chromatic_normalize(crudo))
        VERDAD_SEG.append((gt == 1))

    assert ENTRADAS_SEG, (
        f'Ninguna de las {len(IDS_TEST)} imagenes anotadas se pudo abrir bajo {RAIZ_MASCARAS}. '
        'Revisa que las imagenes acompanen a las anotaciones COCO.')
    ENTRADAS_SEG = np.stack(ENTRADAS_SEG)
    VERDAD_SEG = np.stack(VERDAD_SEG)
    print(f'Held-out del segmentador: {len(VERDAD_SEG)} mascaras de {len(IDS_TEST)} anotadas'
          + (f' ({sin_archivo} sin imagen asociada)' if sin_archivo else ''))

In [ ]:
def mascaras_keras(modelo, entradas, batch=8):
    salida = modelo.predict(entradas.astype(np.float32) / 255.0, batch_size=batch, verbose=0)
    return np.argmax(salida, axis=-1) == 1


def mascaras_tflite(ruta, entradas):
    interp = tf.lite.Interpreter(model_path=str(ruta), num_threads=HILOS)
    interp.allocate_tensors()
    detalle_in = interp.get_input_details()[0]
    detalle_out = interp.get_output_details()[0]
    predichas = []
    for i, imagen in enumerate(entradas):
        dato = (imagen.astype(np.float32) / 255.0)[np.newaxis]
        if detalle_in['dtype'] in (np.uint8, np.int8):
            dato = _cuantizar(dato, detalle_in)
        interp.set_tensor(detalle_in['index'], dato.astype(detalle_in['dtype']))
        interp.invoke()
        logits = _descuantizar(interp.get_tensor(detalle_out['index'])[0], detalle_out)
        predichas.append(np.argmax(logits, axis=-1) == 1)
        if (i + 1) % 50 == 0:
            print(f'      {i + 1}/{len(entradas)}', end='\r')
    return np.stack(predichas)


def metricas_segmentacion(pred, gt):
    inter = float(np.logical_and(pred, gt).sum())
    recall = inter / float(gt.sum()) if gt.sum() else 0.0
    dice = 2 * inter / float(pred.sum() + gt.sum()) if (pred.sum() + gt.sum()) else 0.0
    union = float(np.logical_or(pred, gt).sum())
    return recall, dice, (inter / union if union else 0.0)


tabla_seg = None
if not SEG_DISPONIBLE:
    print('Evaluacion del segmentador omitida: faltan entradas (ver el inventario de la celda 3).')
else:
    import time as _t
    filas_seg = []
    referencia_seg = None
    for variante, clave in [('Keras', 'seg_keras'), ('TFLite float32', 'seg_f32'), ('TFLite int8', 'seg_int8')]:
        ruta = MODELOS[clave]
        inicio = _t.time()
        pred = mascaras_keras(_MSEG, ENTRADAS_SEG) if clave == 'seg_keras' else mascaras_tflite(ruta, ENTRADAS_SEG)
        por_imagen = [metricas_segmentacion(p, g) for p, g in zip(pred, VERDAD_SEG)]
        recall, dice, iou = (float(np.mean([m[k] for m in por_imagen])) for k in range(3))
        if referencia_seg is None:
            referencia_seg = (recall, dice, iou)
        filas_seg.append({'Modelo': 'M_seg (segmentador)', 'Variante': variante,
                          'Tamano (MB)': round(Path(ruta).stat().st_size / 1e6, 2),
                          'Recall de hoja': round(recall, 4), 'Dice': round(dice, 4), 'IoU': round(iou, 4),
                          'D Recall': round(recall - referencia_seg[0], 4),
                          'D Dice': round(dice - referencia_seg[1], 4),
                          'D IoU': round(iou - referencia_seg[2], 4)})
        print(f'  {variante:15s} recall={recall:.4f}  Dice={dice:.4f}  IoU={iou:.4f}'
              f'   [{_t.time() - inicio:.0f}s]')

    tabla_seg = pd.DataFrame(filas_seg)
    tabla_seg.to_csv(OUT / 'evaluacion_variantes_mseg.csv', index=False)
    json.dump(filas_seg, open(OUT / 'evaluacion_variantes_mseg.json', 'w'), indent=2, ensure_ascii=False)
tabla_seg

In [ ]:
for modelo in tabla['Modelo'].unique():
    sub = tabla[tabla['Modelo'] == modelo]
    caida = float(sub[sub['Variante'] == 'TFLite int8']['D F1 macro'].iloc[0])
    if abs(caida) < 0.005:
        juicio = 'la cuantizacion no altera de forma apreciable el desempeno'
    elif abs(caida) < 0.02:
        juicio = 'la cuantizacion introduce una perdida menor, aceptable para despliegue movil'
    else:
        juicio = 'la cuantizacion introduce una perdida relevante que conviene reportar'
    print(f'{modelo}: D F1 macro int8 = {caida:+.4f} -> {juicio}.')

if tabla_seg is None:
    print('\nM_seg: sin evaluar. La observacion 2 del informe queda abierta hasta obtener estas cifras.')
else:
    caida_seg = float(tabla_seg[tabla_seg['Variante'] == 'TFLite int8']['D Recall'].iloc[0])
    if abs(caida_seg) < 0.005:
        juicio_seg = 'la cuantizacion no altera de forma apreciable el recall de hoja'
    elif abs(caida_seg) < 0.02:
        juicio_seg = 'la cuantizacion introduce una perdida menor en el recall de hoja'
    else:
        juicio_seg = 'la cuantizacion degrada el recall de hoja de forma relevante'
    print(f'M_seg (segmentador): D recall int8 = {caida_seg:+.4f} -> {juicio_seg}.')
    print('\nEsta es la variante desplegada del segmentador, por lo que su metrica int8 es la que corresponde reportar.')